#Incremental data ingestion from files

##SQL: COPY INTO method (Legacy)

###Create an empty table with schema

In [0]:
%sql
CREATE OR REPLACE TABLE sql_csv_copyinto (
  customer_id	INT,
  first_name STRING,
  last_name	STRING,
  date_of_birth	DATE,
  gender STRING
);

###Populate table from files using COPY INTO method

In [0]:
%sql
COPY INTO sql_csv_copyinto
FROM '/Volumes/workspace/default/csv_files_source'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');
-- COPY_OPTIONS can be either mergeSchema (=> schema evolution) or force (=> schema override)
-- mergeSchema implies that the schema evolves according to the new incoming data
-- force implies that the schema is overridden by the new incoming data

###Post-checks

In [0]:
%sql
SELECT * FROM sql_csv_copyinto LIMIT 10;

In [0]:
%sql
DESCRIBE EXTENDED sql_csv_copyinto;
-- OBS! Table type: MANAGED != STREAMING

In [0]:
%sql
DESCRIBE HISTORY sql_csv_copyinto;
-- OBS! Operation: COPY INTO

###Manually add a new file to the volume

###Call COPY INTO statement again

In [0]:
%sql
COPY INTO sql_csv_copyinto
FROM '/Volumes/workspace/default/csv_files_source'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

-- OBS!: Incremental ingestion works: 10 inserted rows only

##SQL: Auto Loader method

###Create a Streaming Table

In [0]:
%sql
-- Pre-req: Connect to the Serverless SQL Warehouse to use Auto Loader

CREATE OR REFRESH STREAMING TABLE sql_csv_autoloader
SCHEDULE EVERY 1 WEEK -- Optional
AS
SELECT *
FROM STREAM read_files(
  '/Volumes/workspace/default/csv_files_source',
  format => 'CSV',
  header => true
);

In [0]:
%sql
select * from sql_csv_autoloader;

In [0]:
%sql
describe table extended sql_csv_autoloader; 
-- OBS! Table type: STREAMING

In [0]:
%sql
describe history sql_csv_autoloader;

###Manually add a new 001.csv file to the volume

###Refresh Streaming Table

In [0]:
%sql
REFRESH STREAMING TABLE sql_csv_autoloader;
-- OBS! Incremental ingestion works: 10 inserted rows only

In [0]:
%sql
select * from sql_csv_autoloader;

In [0]:
%sql
describe history sql_csv_autoloader;

###Drop table

In [0]:
%sql
DROP TABLE IF EXISTS sql_csv_autoloader;

##PySpark: Auto Loader method
Data Ingestion from Cloud Storage

###Enable Auto Loader

In [0]:
(spark
 .readStream # enable Auto Loader
 .format('cloudFiles')
 .option('cloudFiles.format', 'csv')
 .option('cloudFiles.schemaLocation', '/Volumes/workspace/default/checkpoint') #chekpoint path. Necessary to track schema inference and evolution
 .load('/Volumes//workspace/default/csv_files_source')
.writeStream # persist the outcome as a table
.option('checkpointLocation', '/Volumes/workspace/default/checkpoint') #chekpoint path. Checkpoint to maintain streaming state and progress
.trigger(availableNow=True)
.toTable('workspace.default.py_csv_autoloader')
)

#ToDo: Play with the different trigger modes. See 4.28 for more info

###Post-check query

In [0]:
%sql
SELECT * FROM py_csv_autoloader;

In [0]:
%sql
DESCRIBE TABLE EXTENDED py_csv_autoloader;
-- OBS! Table type: MANAGED